In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Пути
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

# Настройки визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (10, 5)

# Загружаем train (уже оптимизированный parquet)
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
print(f"Shape: {train.shape}")
print(f"Память: {train.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

In [ ]:
# Конвертируем TransactionDT (секунды от старта) в "день от начала датасета"
train["day"] = (train["TransactionDT"] // 86400).astype(np.int16)

# Группируем по дням: сколько транзакций и какая доля fraud
daily = train.groupby("day", observed=True).agg(
    n_transactions=("isFraud", "size"),
    fraud_rate=("isFraud", "mean"),
    n_fraud=("isFraud", "sum"),
)

print(f"Дней в датасете: {len(daily)}")
print(f"Среднее кол-во транзакций в день: {daily['n_transactions'].mean():.0f}")
print(f"Средняя fraud rate: {daily['fraud_rate'].mean() * 100:.2f}%")
print(f"Min / max fraud rate по дням: "
      f"{daily['fraud_rate'].min() * 100:.2f}% / {daily['fraud_rate'].max() * 100:.2f}%")

# Строим два графика: объём транзакций и fraud rate по дням
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(daily.index, daily["n_transactions"], color="steelblue", linewidth=1)
axes[0].set_ylabel("Транзакций в день")
axes[0].set_title("Объём транзакций по дням", fontsize=12)
axes[0].grid(alpha=0.3)

axes[1].plot(daily.index, daily["fraud_rate"] * 100, color="crimson", linewidth=1)
axes[1].axhline(train["isFraud"].mean() * 100, color="black",
                linestyle="--", alpha=0.5, label="Общее среднее")
axes[1].set_xlabel("День от начала датасета")
axes[1].set_ylabel("Fraud rate, %")
axes[1].set_title("Fraud rate по дням", fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "01_fraud_rate_over_time.png", bbox_inches="tight")
plt.show()

In [ ]:
# Час внутри дня: TransactionDT % 86400 -> секунда в сутках -> час
train["hour"] = ((train["TransactionDT"] % 86400) // 3600).astype(np.int8)

hourly = train.groupby("hour", observed=True).agg(
    n_transactions=("isFraud", "size"),
    fraud_rate=("isFraud", "mean"),
)

fig, ax1 = plt.subplots(figsize=(11, 5))

# Левая ось — количество транзакций (столбцы)
color1 = "steelblue"
ax1.bar(hourly.index, hourly["n_transactions"], color=color1, alpha=0.4,
        label="Транзакций")
ax1.set_xlabel("Час (по TransactionDT, часовой пояс неизвестен)")
ax1.set_ylabel("Кол-во транзакций", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_xticks(range(0, 24))

# Правая ось — fraud rate (линия)
ax2 = ax1.twinx()
color2 = "crimson"
ax2.plot(hourly.index, hourly["fraud_rate"] * 100, color=color2,
         linewidth=2, marker="o", label="Fraud rate")
ax2.axhline(train["isFraud"].mean() * 100, color="black",
            linestyle="--", alpha=0.5)
ax2.set_ylabel("Fraud rate, %", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)
ax2.grid(False)

plt.title("Fraud rate по часам суток (reference-UTC)", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "02_fraud_rate_by_hour.png", bbox_inches="tight")
plt.show()

# Выведем пик и минимум
print(f"Час с максимальной fraud rate: {hourly['fraud_rate'].idxmax()}h "
      f"({hourly['fraud_rate'].max() * 100:.2f}%)")
print(f"Час с минимальной fraud rate: {hourly['fraud_rate'].idxmin()}h "
      f"({hourly['fraud_rate'].min() * 100:.2f}%)")

In [ ]:
# Процент пропусков по каждой колонке
missing = (train.isna().mean() * 100).sort_values(ascending=False)

print(f"Колонок всего: {len(missing)}")
print(f"Без пропусков: {(missing == 0).sum()}")
print(f"Хотя бы один пропуск: {(missing > 0).sum()}")
print(f"Больше 50% пропусков: {(missing > 50).sum()}")
print(f"Больше 90% пропусков: {(missing > 90).sum()}")
print()
print("Топ-15 колонок с самым высоким % пропусков:")
print(missing.head(15).to_string())
print()
print("Колонки без пропусков (первые 20):")
print(missing[missing == 0].head(20).index.tolist())

In [ ]:
# Сортируем колонки по проценту пропусков
missing_sorted = missing.sort_values()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["steelblue" if v < 50 else "orange" if v < 90 else "crimson"
          for v in missing_sorted.values]
ax.bar(range(len(missing_sorted)), missing_sorted.values, color=colors, width=1.0)

ax.axhline(50, color="orange", linestyle="--", alpha=0.5, label="50% пропусков")
ax.axhline(90, color="crimson", linestyle="--", alpha=0.5, label="90% пропусков")

ax.set_xlabel("Колонки (отсортированы по % пропусков)")
ax.set_ylabel("% пропусков")
ax.set_title("Распределение % пропусков по всем колонкам", fontsize=12)
ax.legend()
ax.set_xlim(0, len(missing_sorted))

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "03_missing_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Сколько уникальных уровней % пропусков у V-колонок?
v_cols = [c for c in train.columns if c.startswith("V")]
v_missing = (train[v_cols].isna().mean() * 100).round(2)

unique_levels = v_missing.value_counts().sort_index(ascending=False)
print(f"V-колонок всего: {len(v_cols)}")
print(f"Уникальных уровней % пропусков: {len(unique_levels)}")
print()
print("Топ-10 самых частых уровней (сколько колонок имеют этот % пропусков):")
print(unique_levels.head(10).to_string())

In [ ]:
# Разделим колонки по типам данных
numeric_cols = train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = train.select_dtypes(include=["category", "object"]).columns.tolist()

print(f"Числовых колонок: {len(numeric_cols)}")
print(f"Категориальных колонок: {len(categorical_cols)}")
print()
print("Категориальные колонки (все):")
for c in categorical_cols:
    nunique = train[c].nunique()
    print(f"  {c:25s} — уникальных значений: {nunique}")

In [ ]:
# Ключевые категории для проверки
key_cats = ["ProductCD", "card4", "card6", "DeviceType"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_cats):
    stats = train.groupby(col, observed=True).agg(
        n=("isFraud", "size"),
        fraud_rate=("isFraud", "mean"),
    ).sort_values("n", ascending=False)

    ax = axes[i]
    bars = ax.bar(stats.index.astype(str), stats["fraud_rate"] * 100,
                  color=["steelblue"] * len(stats))
    ax.axhline(train["isFraud"].mean() * 100, color="black",
               linestyle="--", alpha=0.5, label="Среднее (3.5%)")
    ax.set_title(f"{col} (кол-во транзакций в скобках)", fontsize=11)
    ax.set_ylabel("Fraud rate, %")
    ax.legend()

    # Подписи с количеством транзакций
    for bar, (cat, row) in zip(bars, stats.iterrows()):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.1,
                f"{row['n']:,}", ha="center", fontsize=8, rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "04_fraud_rate_by_key_cats.png", bbox_inches="tight")
plt.show()

In [ ]:
# Топ-20 email-доменов по количеству транзакций
email_stats = train.groupby("P_emaildomain", observed=True).agg(
    n=("isFraud", "size"),
    fraud_rate=("isFraud", "mean"),
).sort_values("n", ascending=False).head(20)

# Сортируем по fraud rate для наглядности
email_stats = email_stats.sort_values("fraud_rate")

fig, ax = plt.subplots(figsize=(12, 7))
colors = ["steelblue" if v < 0.05 else "orange" if v < 0.10 else "crimson"
          for v in email_stats["fraud_rate"]]
ax.barh(email_stats.index.astype(str), email_stats["fraud_rate"] * 100, color=colors)
ax.axvline(train["isFraud"].mean() * 100, color="black",
           linestyle="--", alpha=0.5, label="Среднее (3.5%)")

# Подписи с количеством транзакций справа от столбца
for i, (idx, row) in enumerate(email_stats.iterrows()):
    ax.text(row["fraud_rate"] * 100 + 0.2, i,
            f"n={row['n']:,}", va="center", fontsize=8)

ax.set_xlabel("Fraud rate, %")
ax.set_title("Fraud rate по топ-20 email-доменам (P_emaildomain)", fontsize=12)
ax.legend()

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "05_fraud_rate_by_email.png", bbox_inches="tight")
plt.show()

In [ ]:
ideas_text = """# IEEE-CIS Fraud Detection — Ideas & Findings

## 1. Target

- **isFraud = 3.6%** overall, heavy imbalance → use ROC-AUC, PR-AUC; stratify where sensible.
- **Fraud rate drifts over time** (1.1%–7.0% across 182 days): ruling out random KFold, only time-based CV.
- First ~25 days have **lower fraud rate + higher volume** (likely holiday season in anonymized data).

## 2. Time features

- Strongest hour-of-day signal: peak fraud at hour 7 (**10.6%**), minimum at hour 13 (**2.3%**), 4.6× spread.
- Derived features to engineer in day 7:
  - `hour = (TransactionDT % 86400) // 3600`
  - `day = TransactionDT // 86400`
  - `is_night = hour in [0..10]`

## 3. Missing values

- 22 of 436 columns have zero missing; 214 columns have >50%; 12 have >90%.
- **Strong missing-value clustering**: 339 V-columns group into only **14 distinct missing-rate levels**.
  - Largest cluster: 46 V-cols with exactly 77.91% missing — almost certainly a single Vesta subsystem.
  - Implication: within each cluster, V-cols are highly correlated. Reduce via correlation > 0.98 per group.
- Identity columns ~76% missing because `identity` table only covers 24% of transactions.
- Action items:
  - Binary features: `has_identity`, `v_block_77_missing` etc.
  - LightGBM/CatBoost handle NaN natively — no imputation for tree baselines.

## 4. Categorical features

### Low-cardinality (2-5 values) — 22 cols
`ProductCD, card4, card6, DeviceType, M1..M9, most id_*`

- `ProductCD`: W=2% vs C=11.6% — 5× difference, strong signal.
- `card6`: credit 6.6% vs debit 2.5% — 2.5× difference.
- `DeviceType`: mobile 10% vs desktop 6.5% — plus the mere fact of having identity = risk signal.
- Encoding: label encoding or native `categorical_feature` in LightGBM.

### Medium-cardinality (50–130) — `P_emaildomain`, `R_emaildomain`, `id_30`, `id_31`
- `P_emaildomain` fraud rate spans 0.4% (sbcglobal, att) to 9.3% (outlook) — 25× range.
- Encoding strategy:
  - **Frequency encoding** (safe, computed on train+test together).
  - **OOF target encoding** with smoothing α=10..50 and **expanding window** along time to avoid both self-leakage and time leakage.
  - Additional bool features: `is_ms_email`, `is_telecom_email`.
  - Split domain into provider + suffix (`yahoo.com.mx` → `yahoo` + `com.mx`).

### High-cardinality (260+) — `id_33`, `DeviceInfo` (1786 unique)
- `DeviceInfo` needs heavy cleanup: extract vendor prefix (`SAMSUNG SM-G892A ...` → `SAMSUNG`).
- `id_33` (screen resolution `1334x750`): parse width × height → two numeric features; also `aspect_ratio`.
- After cleanup: frequency / target encoding.

## 5. Validation strategy (day 4)

- **TimeSeriesKFold** by `TransactionDT`, 5 folds, each ~36 days.
- Alternative sanity check: `GroupKFold` by `card1` (proxy for user).
- Encoding computations **inside each fold only** (no global leakage).
- Sanity check: CV AUC should match Public LB within ±0.005–0.01. If not — CV is wrong.

## 6. Model iteration plan

- v0 baseline: LightGBM, raw features + native categoricals, time-based CV. Target: CV AUC ~0.91.
- v1: add hour/day features, card1 aggregations. Target: ~0.93.
- v2: frequency + target encoding (expanding-window). Target: ~0.94.
- v3: D1 trick (D1 - day = user registration proxy), device info cleanup. Target: ~0.945.
- v4: LightGBM + CatBoost rank-average ensemble. Target: ~0.95.
"""

ideas_path = PROJECT_ROOT / "reports" / "ideas.md"
ideas_path.write_text(ideas_text, encoding="utf-8")
print(f"Saved to: {ideas_path}")
print(f"Size: {ideas_path.stat().st_size} bytes")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
ideas_path = PROJECT_ROOT / "reports" / "ideas.md"

insights_from_top = """

## 7. Insights from top solutions (after day 0 reading)

### 7.1 UID reconstruction (Chris Deotte, 1st place technique)

- Dataset does NOT contain a client ID (anonymized by Vesta).
- Reconstructed UID: `card1 + addr1 + D1n`, where `D1n = D1 - day`.
- `D1n` is a per-client constant (proxy for card registration day relative to dataset start).
- Why it works: transactions from the same client share this tuple across time.

### 7.2 Aggregations by UID

Once UID is reconstructed, create per-UID aggregations over transaction history:

- `uid_n_transactions` — total transactions by this UID
- `uid_amt_mean`, `uid_amt_std`, `uid_amt_max` — spending profile
- `uid_n_unique_products` — diversity of product types
- `uid_n_unique_addr2` — geography diversity (multiple countries = risk signal)
- `uid_first_txn_day`, `uid_last_txn_day` — activity window

These are among the strongest features in the competition.

### 7.3 Normalized D-columns

- D-columns (D1-D15) are timedelta-like: days since some event.
- For client-level features, subtract `day` from each D column to get the **event date**, not the delta.
- Example: `D1n = D1 - day` → day the card was issued (constant per client).
- Apply similar normalization to D2-D15 for additional client-invariant features.

### 7.4 Ensemble strategy (1st place)

- Three diverse GBDT models: XGBoost + CatBoost + LightGBM.
- Blended with near-equal weights (avoids LB overfitting).
- Trained on different feature sets to increase diversity.
- Final private LB AUC: 0.9459.

### 7.5 What to realistically target for this project

- v0 baseline (raw features, LightGBM only): CV ~0.91, Public LB ~0.91
- v1-v3 (hour/day, aggregations, encodings, D-normalization): CV ~0.94
- v4 (add UID aggregations + simple XGB+CatBoost ensemble): CV ~0.945, Private LB ~0.92 = top 20%

Top 5-10% requires more aggressive UID reconstruction + client-level feature engineering.
Top 1% adds V-column grouping + custom post-processing.
"""

# Добавляем к существующему ideas.md
with open(ideas_path, "a", encoding="utf-8") as f:
    f.write(insights_from_top)

print(f"Updated: {ideas_path}")
print(f"New size: {ideas_path.stat().st_size} bytes")